# Step 3: Utilities (`utils/timer.py`)

**Goal:** Two small tools used by every API client:
1. `Timer` — measures how long an API call takes
2. `calculate_cost()` — converts token counts into a USD cost

---

## Why a separate utils file?

Both the OpenAI, Gemini, and Claude clients will need to:
- Record how many seconds the API took to respond
- Calculate the cost of each call in USD

Instead of copy-pasting that logic into each of the three client files,
we write it once here and import it everywhere.
This is the **DRY principle** — Don't Repeat Yourself.

---

## What is a context manager?

You've already used context managers — `open()` is the classic example:

```python
with open("file.txt") as f:
    data = f.read()
# file is automatically closed here, even if an error occurred
```

The `with` block automatically runs **setup** on entry and **teardown** on exit.
We're building the same pattern for timing:

```python
with Timer() as t:
    response = call_openai(...)   # whatever is inside gets timed

print(t.elapsed)  # seconds the call took
```

Python calls `__enter__` when you enter the `with` block and `__exit__` when you leave it
(regardless of whether an exception was raised).

---

## The `Timer` class

In [ ]:
import time

class Timer:
    """Context manager that measures elapsed wall-clock time."""

    def __init__(self):
        self.elapsed: float = 0.0  # seconds, available after __exit__

    def __enter__(self) -> "Timer":
        self._start = time.perf_counter()
        return self

    def __exit__(self, *_) -> None:
        self.elapsed = time.perf_counter() - self._start

### Design decisions:

**`time.perf_counter()` not `time.time()`**  
`time.time()` returns wall-clock time in seconds, but it can jump backwards if the system clock
is adjusted (e.g. NTP sync). `perf_counter()` is a monotonic, high-resolution counter — it only
ever goes forward, and it's more precise. Always use it for measuring elapsed time.

**`*_` in `__exit__`**  
`__exit__` always receives three arguments: the exception type, value, and traceback.
We don't use any of them — we just want to record the end time.
The `*_` pattern is Python's convention for "I know these exist; I'm intentionally ignoring them."
Using `*args` would trigger a linting warning ("args is not accessed").

**`return self` in `__enter__`**  
This is what makes `as t` work. Without it, `t` would be `None`.

In [ ]:
# See it in action
import time

with Timer() as t:
    time.sleep(0.3)   # simulating an API call that takes 300ms

print(f"Elapsed: {t.elapsed:.3f}s")  # should be ~0.300

---

## Why do input and output tokens have different prices?

This is a great question. The short answer: **output is more expensive to generate than input is to read.**

- **Input tokens** — the model reads your prompt. This is a forward pass, relatively fast.
- **Output tokens** — the model generates each token one at a time, autoregressively.
  Generating 500 tokens literally runs the model 500 times.

This is why output typically costs 3–5x more per token than input.
Example with Claude Sonnet 4.6:

| | Rate | 1,000 tokens |
|---|---|---|
| Input  | $3.00 / 1M | $0.003 |
| Output | $15.00 / 1M | $0.015 |

We already store both rates in `ModelConfig.input_cost_per_1k` and `ModelConfig.output_cost_per_1k`.

---

## `calculate_cost()`

In [ ]:
from dataclasses import dataclass, field

# Minimal ModelConfig for illustration (same shape as the real one in config.py)
@dataclass(frozen=True)
class ModelConfig:
    provider: str
    model_id: str
    display_name: str
    input_cost_per_1k: float
    output_cost_per_1k: float
    supports_vision: bool = field(default=False)


def calculate_cost(model: ModelConfig, input_tokens: int, output_tokens: int) -> float:
    """Return the total USD cost for a single API call.

    Providers charge input and output tokens at different rates, so we
    calculate each separately and sum them.

    Args:
        model:         ModelConfig for the model that was called.
        input_tokens:  Number of tokens in the prompt (what you sent).
        output_tokens: Number of tokens in the response (what came back).

    Returns:
        Total cost in USD, rounded to 6 decimal places.
    """
    input_cost  = (input_tokens  / 1000) * model.input_cost_per_1k
    output_cost = (output_tokens / 1000) * model.output_cost_per_1k
    return round(input_cost + output_cost, 6)

### How the formula works:

The rates in `ModelConfig` are **per 1K tokens** (matching how providers publish pricing).
So to get the cost for any number of tokens:

```
cost = (tokens / 1000) * rate_per_1k
```

We do this separately for input and output, then sum.

**Why `round(..., 6)`?**  
Floating-point arithmetic can produce results like `0.00300000000000001`.
Rounding to 6 decimal places ($0.000001 = one hundredth of a cent) is precise
enough for cost tracking and keeps the numbers clean for display.

In [ ]:
# Example: Claude Sonnet 4.6 — 500 input tokens, 200 output tokens
sonnet = ModelConfig(
    provider="claude",
    model_id="claude-sonnet-4-6",
    display_name="Claude Sonnet 4.6",
    input_cost_per_1k=0.003000,   # $3.00 / 1M
    output_cost_per_1k=0.015000,  # $15.00 / 1M
    supports_vision=True,
)

cost = calculate_cost(sonnet, input_tokens=500, output_tokens=200)
print(f"Cost: ${cost:.6f}")
# input:  (500/1000) * 0.003 = 0.001500
# output: (200/1000) * 0.015 = 0.003000
# total:  0.004500

In [ ]:
# Free tier model — Gemini 2.0 Flash Lite (both rates are 0.0)
flash_lite = ModelConfig(
    provider="gemini",
    model_id="gemini-2.0-flash-lite",
    display_name="Gemini 2.0 Flash Lite",
    input_cost_per_1k=0.0,
    output_cost_per_1k=0.0,
)

cost = calculate_cost(flash_lite, input_tokens=1000, output_tokens=1000)
print(f"Cost: ${cost:.6f}")  # $0.000000

---

## How Timer and calculate_cost work together

In the API clients we'll build next, every call will look like this:

```python
with Timer() as t:
    response = client.chat(...)         # actual API call

cost = calculate_cost(
    model=model_config,
    input_tokens=response.usage.input_tokens,
    output_tokens=response.usage.output_tokens,
)

# Both t.elapsed and cost go into our ChatResponse
```

The `ChatResponse` Pydantic model (from Session 1) already has `latency_ms` and `cost_usd` fields
waiting for exactly this data.

---

## Summary

| What | Why |
|---|---|
| Context manager (`with` block) | Automatic start/stop — no way to forget to stop the timer |
| `perf_counter()` not `time.time()` | Monotonic, high-resolution — immune to clock adjustments |
| `*_` in `__exit__` | Signals intentional ignore of exception args; keeps linter happy |
| Separate input/output costs | Providers charge them at different rates — 3–5x difference is common |
| `round(..., 6)` | Avoids floating-point noise; $0.000001 precision is more than enough |

**Next step:** `services/openai_client.py` — first real API call, using Timer and calculate_cost.